# Trabalho 2 — 03 · RAG Chatbot fundamentado
### Pokédex Inteligente · 151 Pokémon de Kanto / FireRed

**Objetivo:** usar o índice TF-IDF do Notebook 02 para recuperar documentos e, **somente depois**, utilizar um LLM para formular respostas em português com evidências auditáveis.

**Pipeline acadêmico:** Trabalho 1 (corpus + PLN tradicional) → Notebook 01 (documentos temáticos) → Notebook 02 (TF-IDF + avaliação) → **Notebook 03 (RAG + avaliação da geração)**.

**Restrições:** sem novo treinamento, sem nova coleta, sem indexação e sem busca web. Modelo generativo **não substitui** o corpus. Uma resposta textual do LLM não é automaticamente considerada factual: o sistema exige citações a documentos efetivamente recuperados e trechos literalmente presentes neles. Mesmo assim, validação de IDs e citações **não prova** que a afirmação seja logicamente sustentada; esse aspecto será objeto de revisão humana.

> **Execução:** o FLAN-T5 roda localmente, sem chave de API. A geração carrega os pesos do cache local quando a caixa de geração é marcada. Os experimentos sem corpus são **exclusivos de avaliação** e nunca alimentam o chatbot de produção.
**Atualização:** memória de múltiplos Pokémon, pedidos de esclarecimento e respostas contextuais a limites.


## 0 · Configuração e diretórios
Usamos os mesmos diretórios do Notebook 02. Nenhuma chave de API é necessária: o modelo `google/flan-t5-small` é carregado do cache local. Fora do Colab, use `PLN_AVALIACAO_DIR` apontando para a pasta `avaliacao`.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from IPython.display import display, Markdown
import hashlib
import json
import logging
import os
import re
import unicodedata
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy import sparse

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

def resolve_workspace_root():
    """Resolve a raiz no Colab, VS Code ou execução por linha de comando."""
    if IN_COLAB:
        return Path('/content/drive/MyDrive/IFSP-ADS/ultimo_periodo/tc_2/avaliacao')
    configured = os.environ.get('PLN_AVALIACAO_DIR', '').strip()
    if configured:
        return Path(configured).expanduser().resolve()
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'corpus').exists() or (candidate / 'application').exists():
            return candidate
    return Path.cwd().resolve()

AVALIACAO_DIR = resolve_workspace_root()
APPLICATION_DIR = AVALIACAO_DIR / 'application'
DOC_DIR = APPLICATION_DIR / 'data' / 'rag' / 'documents'
ARTIFACTS_DIR = APPLICATION_DIR / 'artifacts' / 'retrieval'
REPORTS_DIR = APPLICATION_DIR / 'reports'
LOGS_DIR = APPLICATION_DIR / 'logs'
EVAL_DIR = APPLICATION_DIR / 'data' / 'evaluation'
for directory in (REPORTS_DIR, LOGS_DIR, EVAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DOCUMENTS_PATH = DOC_DIR / 'pokemon_rag_documents.jsonl'
RECORDS_PATH = DOC_DIR / 'pokemon_rag_records.jsonl'
MANIFEST_PATH = ARTIFACTS_DIR / 'retrieval_manifest.json'
RUN_AT = datetime.now(timezone.utc).isoformat()

logger = logging.getLogger('pokemon_rag_chatbot')
logger.handlers.clear()
logger.setLevel(logging.INFO)
logger.propagate = False
logger.addHandler(logging.FileHandler(LOGS_DIR / '03_rag_chatbot.log', encoding='utf-8'))
# logger.addHandler(logging.StreamHandler())
print('Documentos:', DOCUMENTS_PATH)
print('Índices do Notebook 02:', ARTIFACTS_DIR)


## 1 · Abrir somente os artefatos persistidos no Notebook 02
Conferimos hash SHA-256 do JSONL, ordenação dos IDs e dimensões. Se os documentos mudaram, é necessário reexecutar o Notebook 02. Só carregue arquivos `.joblib` gerados pelo seu próprio projeto: deserialização Joblib de origem desconhecida é insegura.

In [ ]:
def read_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f'Arquivo não encontrado: {path}. Execute os notebooks anteriores.')
    with path.open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'Manifesto não encontrado: {MANIFEST_PATH}. Execute a exportação do Notebook 02.'
    )
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
required_manifest = {'source_document_sha256', 'document_ids', 'models', 'query_normalization_rules'}
missing = required_manifest - manifest.keys()
if missing:
    raise ValueError(f'Manifesto incompleto. Campos ausentes: {sorted(missing)}')

actual_sha = hashlib.sha256(DOCUMENTS_PATH.read_bytes()).hexdigest()
if actual_sha != manifest['source_document_sha256']:
    raise ValueError(
        'O documento JSONL foi alterado desde o ajuste do TF-IDF. '
        'Reexecute o Notebook 02 com os documentos atuais antes de gerar respostas.'
    )

raw_documents = read_jsonl(DOCUMENTS_PATH)
records = read_jsonl(RECORDS_PATH)
by_id = {d['document_id']: d for d in raw_documents}
if len(by_id) != len(raw_documents):
    raise ValueError('Existem document_id duplicados no JSONL.')

ordered_ids = manifest['document_ids']
if set(ordered_ids) != set(by_id) or len(ordered_ids) != len(raw_documents):
    raise ValueError('IDs de documentos no manifesto não correspondem ao JSONL atual.')
documents = [by_id[doc_id] for doc_id in ordered_ids]
if len({int(r['id']) for r in records}) != 151:
    raise ValueError('Esperados 151 Pokémon (IDs únicos) na Pokédex de Kanto.')

model_info = manifest['models']['tfidf']
vectorizer_path = ARTIFACTS_DIR / model_info['vectorizer']
matrix_path = ARTIFACTS_DIR / model_info['matrix']
for p in (vectorizer_path, matrix_path):
    if not p.is_file():
        raise FileNotFoundError(f'Índice ausente: {p}. Reexecute o Notebook 02.')

vectorizer = joblib.load(vectorizer_path)  # apenas artefatos gerados pelo próprio projeto
index_matrix = sparse.load_npz(matrix_path)
if index_matrix.shape != (len(documents), len(vectorizer.get_feature_names_out())):
    raise ValueError('Dimensões do índice incompatíveis com documentos/vetorizador.')

DOC_LOOKUP = {d['document_id']: d for d in documents}
POKEMON_NAMES = {str(r.get('name', r.get('api_name',''))).casefold() for r in records}
print(f'{len(records)} Pokémon | {len(documents)} documentos temáticos | TF-IDF {index_matrix.shape}')
logger.info('Artefatos carregados e verificados | documentos=%d | checksum_ok', len(documents))

## 2 · Normalização idêntica ao Notebook 02
Importamos o arquivo de equivalências sem recalcular vocabulários ou reexecutar o Trabalho 1.

In [ ]:
rules_path = ARTIFACTS_DIR / manifest['query_normalization_rules']
if not rules_path.is_file():
    raise FileNotFoundError(f'Regras ausentes: {rules_path}')
normalization_config = json.loads(rules_path.read_text(encoding='utf-8'))
ALIAS_RULES = [(item['from'], item['to']) for item in normalization_config['rules']]

def normalize_plain(text):
    text = unicodedata.normalize('NFKD', str(text or '').casefold())
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace('-', ' ')
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9\s]', ' ', text)).strip()

def normalize_query(text):
    text = normalize_plain(text)
    for source, target in ALIAS_RULES:
        text = re.sub(r'(?<!\w)' + re.escape(source) + r'(?!\w)', target, text)
    return re.sub(r'\s+', ' ', text).strip()

assert normalize_query('Pokémon do tipo pedra').endswith('type rock')
assert normalize_query('Pedra evolutiva para Eevee').startswith('evolution stone')
assert normalize_query('Rock type Pokémon').endswith('rock type pokemon')
print('Normalização compartilhada carregada do Notebook 01/02.')


## 3 · Recuperador TF-IDF existente
Usa **o índice TF-IDF do Notebook 02** e similaridade de cosseno. Nenhum `fit()` ou nova coleta é executado aqui.

In [ ]:
MAX_INPUT_TOKENS = 384
MAX_DOC_CHARS = 3500

def prompt_text(question, context):
    return ("Responda apenas com base no contexto. Se faltar informação, responda Não sei. "
            "Responda no idioma da pergunta, sem inventar fatos.\n"
            f"Contexto:\n{context}\n\nPergunta: {question}\nResposta:")

def token_count(tokenizer, text):
    encoded = tokenizer(text, add_special_tokens=True, truncation=False)
    ids = encoded.input_ids if hasattr(encoded, 'input_ids') else encoded['input_ids']
    return len(ids[0] if ids and isinstance(ids[0], (list, tuple)) else ids)

def build_context(rows, question=None, tokenizer=None, max_chars=13500, max_doc_chars=MAX_DOC_CHARS):
    parts, used, total = [], [], 0
    for row in rows:
        header = f"FONTE [{row['document_id']}] | Pokémon: {row['pokemon_name']} | seção: {row['section']}\n"
        content = str(row['content'])
        candidate = '\n\n'.join(parts + [header + content])
        if question is not None and tokenizer is not None:
            if token_count(tokenizer, prompt_text(question, candidate)) > MAX_INPUT_TOKENS:
                continue
        else:
            budget = min(max_doc_chars, max_chars - total - len(header) - 2)
            if budget < 100:
                break
            content = content[:budget]
            candidate = '\n\n'.join(parts + [header + content])
        parts.append(header + content)
        used.append({**row, 'content': content})
        total += len(header) + len(content) + 2
    return '\n\n'.join(parts), used

print('Contexto FLAN-T5: orçamento de entrada=', MAX_INPUT_TOKENS, 'tokens; seleção sem truncamento silencioso.')


## 4 · Política de recuperação e interpretação
O ranking é calculado pelo TF-IDF persistido no Notebook 02. Metadados dos documentos orientam a seleção da seção, uma fonte é reservada por espécie em perguntas múltiplas e regras conservadoras identificam limites e perguntas que necessitam de critério.


In [ ]:
import difflib
import re

REFUSAL = (
    "Não encontrei informações suficientes no corpus "
    "de Kanto/FireRed para responder com segurança."
)

NAME_TO_ID = {}
for record in records:
    name = str(record.get("api_name") or record.get("name") or "").strip()
    if name:
        NAME_TO_ID[normalize_plain(name)] = int(record["id"])

KNOWN_NAMES = sorted(NAME_TO_ID, key=len, reverse=True)
ONE_WORD_NAMES = sorted(n for n in KNOWN_NAMES if len(n.split()) == 1 and len(n) >= 4)

COMMON_WORDS = {
    "sobre", "como", "quem", "quais", "qual",
    "quando", "onde", "porque", "para", "mais",
    "menos", "evoluir", "evolucao", "evolucoes",
    "evolui", "pokemon", "pokemons", "pokedex",
    "fire", "red", "habilidade", "habilidades",
    "estatisticas", "ataques", "golpes",
    "tipos", "tipo", "fogo", "agua", "voador",
    "pedra", "eletrico", "nome", "explique",
    "quero", "me", "diga", "have", "tell",
    "what", "which", "where", "from", "about",
    "level", "type", "evolution"
}

def canonicalize_question(question):
    tokens = normalize_plain(question).split()
    result = []
    corrections = []
    for token in tokens:
        if token in NAME_TO_ID or token in COMMON_WORDS or len(token) < 4:
            result.append(token)
            continue
        choices = difflib.get_close_matches(token, ONE_WORD_NAMES, n=2, cutoff=0.84)
        if choices:
            top = choices[0]
            first = difflib.SequenceMatcher(None, token, top).ratio()
            second = difflib.SequenceMatcher(None, token, choices[1]).ratio() if len(choices) > 1 else 0.0
            if first - second >= 0.06 and abs(len(token) - len(top)) <= 2:
                result.append(top)
                corrections.append({"from": token, "to": top})
                continue
        result.append(token)
    return "  ".join(result), corrections

OUT_OF_SCOPE_PATTERNS = [
    r"\b(ginasio|ginasios|gym|gyms|elite four|elite 4|detonado|walkthrough|historia principal|storyline)\b"
]

SECTION_TERMS = [
    ("type_matchups", (
        "fraqueza", "fraquezas", "fraco contra",
        "resistencia", "resistencias",
        "imunidade", "imune", "weakness",
        "weak", "resistant", "effectiveness",
        "super effective"
    )),
    ("evolution", (
        "evolucao", "evolucoes", "evolui", "evoluir", "evoluiria",
        "evolves", "evolve", "evolution", "pedra evolutiva",
        "pre evolucao", "pre-evolucao", "preevolucao", "de quem evolui",
        "forma inicial", "forma evoluida", "evolucao anterior",
        "anterior form", "pre-evolution", "previous form", "evolved from"
    )),
    ("encounters_strict", (
        "onde encontro na natureza", "encontra na natureza",
        "onde aparece selvagem", "onde aparece na natureza"
    )),
    ("encounters", (
        "onde encontro", "onde encontrar", "onde capturar", "onde aparece",
        "where to find", "where catch", "wild encounter", "localizacao selvagem",
        "localizacao", "captura", "capturar", "encontrar", "localizar"
    )),
    ("obtainment", (
        "como obter", "como conseguir", "como ganhar", "como receber", "obter", "obtain", "gift", "troca",
        "disponivel no jogo", "available in firered", "ganha", "recebe"
    )),
    ("stats", (
        "velocidade", "speed", "ataque", "attack", "defesa", "defense",
        "hp", "stat", "estatistica", "estatisticas", "atributo", "atributos",
        "atributos base", "atributos-base", "status", "mais rapido", "mais forte"
    )),
    ("abilities", (
        "habilidade", "habilidades", "ability", "abilities"
    )),
    ("moves", (
        "golpe", "golpes", "movimento", "movimentos", "move", "moves",
        "tm", "hm", "aprende", "learns"
    )),
    ("profile", (
        "tipo", "tipos", "types", "type",
        "peso", "weight", "altura", "height",
        "habitat", "descricao", "description",
        "pokedex", "quem e", "o que e",
        "me explique", "explique", "fale sobre",
        "me fale", "conte sobre", "me conte",
        "informacoes sobre", "tell me about",
        "what is", "who is", "overview"
    ))
]

def explicitly_mentioned_ids(question):
    q, _ = canonicalize_question(question)
    return list(dict.fromkeys(
        NAME_TO_ID[name]
        for name in KNOWN_NAMES
        if re.search(r"(?<!\w)" + re.escape(name) + r"(?!\w)", q)
    ))

def requested_section(question):
    q = normalize_plain(question)
    if re.search(r'\b(?:evolu\w*|evolv\w*|pre[- ]?evolu\w*)\b', q):
        return 'evolution'
    if re.search(r'\b(?:onde|where)\b.*\b(?:encontr\w*|captur\w*|peg\w*|find|catch)\b', q):
        return 'encounters'
    if re.search(r'\b(?:como|how)\b.*\b(?:encontr\w*|captur\w*|peg\w*|find|catch)\b', q):
        return 'encounters'
    if re.search(r'\b(?:encontr\w*|localiz\w*|captur\w*|peg\w*|find|catch)\b', q):
        return 'encounters'
    if re.search(r'\b(?:obter|obten\w*|conseguir|receber|ganhar|obtain|gift|trade|get)\b', q):
        return 'obtainment'
    for section, expressions in SECTION_TERMS:
        if any(re.search(r'(?<!\w)' + re.escape(expr) + r'(?!\w)', q) for expr in expressions):
            return 'encounters' if section == 'encounters_strict' else section
    return None
def scope_gate(question):
    """Regras conservadoras anteriores ao TF-IDF para limites conhecidos do corpus."""
    q = normalize_plain(question)
    if any(re.search(pattern, q) for pattern in OUT_OF_SCOPE_PATTERNS):
        return "out_of_scope"
    # A base carrega apenas os registros de números 001 a 151.
    dex = re.search(r"\b(?:pokemon|pokedex|numero|number|n|no)\s*(?:de\s+numero\s*)?(\d{1,4})\b", q)
    if dex and int(dex.group(1)) not in range(1, 152):
        return "outside_kanto_dex"
    if re.search(r"\b(?:mais forte|melhor pokemon|strongest pokemon|best pokemon)\b", q) and not re.search(
        r"\b(?:ataque|defesa|velocidade|hp|attack|defense|speed|stat|atributo)\b", q
    ):
        return "needs_comparison_criterion"
    # Reconhece pedido de localização de uma espécie nomeada mas ausente no corpus.
    if not explicitly_mentioned_ids(question):
        location = re.search(
            r"\b(?:onde (?:encontro|encontrar|capturo|capturar|acho)|"
            r"como (?:consigo|conseguir|obtenho|obter)|"
            r"where (?:can i )?(?:find|catch)|how (?:can i |to )?get)\s+"
            r"(?:(?:um|uma|o|a|an|the)\s+)?([a-z0-9-]+)\s*$", q
        )
        if location and location.group(1) not in {"pokemon", "pokemons", "monstro", "pokemonzinho"}:
            return "unknown_species"
    return None


def out_of_scope(question):
    return scope_gate(question) in {"out_of_scope", "outside_kanto_dex"}


def same_type_evidence(question, targets, top_k):
    if len(targets) != 1:
        return [], 'same_type_requires_one_reference'
    target_id = targets[0]
    profiles = [row for row in documents if row['section'] == 'profile']
    target = next((row for row in profiles if int(row['pokemon_id']) == target_id), None)
    if not target:
        return [], 'missing_target_profile'
    match = re.search(r'Types in Generation III:\s*([^.]*)\.', str(target['content']), re.IGNORECASE)
    target_types = frozenset(normalize_plain(value) for value in match.group(1).split(',')) if match else frozenset()
    if not target_types:
        return [], 'missing_target_types'
    matches = [row for row in profiles if int(row['pokemon_id']) != target_id and
               (found := re.search(r'Types in Generation III:\s*([^.]*)\.', str(row['content']), re.IGNORECASE)) and
               frozenset(normalize_plain(value) for value in found.group(1).split(',')) == target_types]
    matches.sort(key=lambda row: (int(row['pokemon_id']), row['document_id']))
    return [dict(row, rank=index + 1, score=float(row.get('score', 0.0)))
            for index, row in enumerate([target] + matches[:max(1, top_k - 1)])], 'ready'

def select_evidence(question, top_k=5):
    gate = scope_gate(question)
    if gate:
        return [], gate

    canonical, _ = canonicalize_question(question)
    targets = explicitly_mentioned_ids(canonical)
    section = requested_section(canonical)
    if re.search(r'\b(?:mesm[oa]s?|igual(?:es)?|same)\b.*\b(?:tipo|types?)\b', normalize_plain(question)):
        return same_type_evidence(question, targets, top_k)
    if targets and section is None:
        section = "profile"

    # TF-IDF existente continua sendo o recuperador: sem embeddings e sem novo fit.
    ranked = retrieve(canonical, top_k=len(documents))
    if not ranked:
        return [], "no_lexical_match"
    if targets:
        ranked = [doc for doc in ranked if int(doc["pokemon_id"]) in targets]

    if section == "encounters_strict":
        ranked = [doc for doc in ranked if doc["section"] == "encounters"]
    elif section in ("obtainment", "encounters"):
        ranked = [doc for doc in ranked if doc["section"] in ("obtainment", "encounters", "evolution")]
    elif section:
        ranked = [doc for doc in ranked if doc["section"] == section]

    if not ranked:
        return [], "missing_target_section" if section else "no_relevant_documents"

    if targets:
        missing = set(targets) - {int(d["pokemon_id"]) for d in ranked}
        if missing:
            return [], "missing_comparison_evidence"

        # Primeiro reserva uma fonte para CADA Pokémon solicitado.
        # Em seguida, inclui as fontes adicionais com maior similaridade.
        prioritized = []
        chosen_ids = set()
        for target in targets:
            candidate = next((d for d in ranked if int(d["pokemon_id"]) == target), None)
            if candidate and candidate["document_id"] not in chosen_ids:
                prioritized.append(candidate)
                chosen_ids.add(candidate["document_id"])
        prioritized.extend(d for d in ranked if d["document_id"] not in chosen_ids)
        ranked = prioritized

    selected_rows = [dict(row, rank=idx + 1) for idx, row in enumerate(ranked[:max(int(top_k), len(targets))])]
    return selected_rows, "ready"

print("Recuperação TF-IDF: escopo, várias espécies e cobertura individual configurados.")


## 5 · Montagem do contexto e contrato da resposta
O FLAN-T5 recebe somente a **pergunta**, **instruções de fundamentação** e os **documentos escolhidos**. A saída deve ser JSON com `answer`, `can_answer` e `evidence` (lista de `document_id` e uma **citação literal** do documento). Os documentos são dados, não instruções.

**Nota:** validação de citações e trechos é um controle mecânico, não um verificador semântico. A resposta pode continuar imprecisa mesmo após passar pelo controle.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query, top_k=5):
    query_vec = vectorizer.transform([normalize_query(query)])
    similarities = cosine_similarity(query_vec, index_matrix).flatten()
    ranked_indices = np.argsort(similarities)[::-1]

    results = []
    for idx in ranked_indices:
        score = similarities[idx]
        if score <= 0:
            continue
        doc = documents[idx].copy()
        doc['score'] = float(score)
        results.append(doc)
        if len(results) >= top_k:
            break
    return results

# Realiza o assert de sanidade com o build_context da célula 8 (que não redefine nada aqui)
ctx, chosen = build_context(select_evidence('Qual é a evolução de Eevee?', top_k=5)[0])
assert ctx and any('evolution' in d['document_id'] for d in chosen)
print('Recuperador e teste de sanidade prontos utilizando definições limpas da célula 8.')

## 6 · FLAN-T5 local como gerador


In [ ]:
import os
import time

FLAN_MODEL_NAME = os.environ.get('FLAN_MODEL_NAME', 'google/flan-t5-small')
DEVICE = 'cpu'
FLAN_TOKENIZER = None
FLAN_MODEL = None

def load_flan_t5():
    global FLAN_TOKENIZER, FLAN_MODEL, DEVICE
    if FLAN_TOKENIZER is not None and FLAN_MODEL is not None:
        return FLAN_TOKENIZER, FLAN_MODEL
    try:
        import torch
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    except ImportError as error:
        raise RuntimeError('Instale transformers, sentencepiece e torch para usar FLAN-T5.') from error
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    FLAN_TOKENIZER = AutoTokenizer.from_pretrained(FLAN_MODEL_NAME, local_files_only=True)
    FLAN_MODEL = AutoModelForSeq2SeqLM.from_pretrained(FLAN_MODEL_NAME, local_files_only=True).to(DEVICE).eval()
    return FLAN_TOKENIZER, FLAN_MODEL

def generate_with_flan_t5(question, context):
    tokenizer, model = load_flan_t5()
    prompt = prompt_text(question, context)
    input_tokens = token_count(tokenizer, prompt)
    if input_tokens > MAX_INPUT_TOKENS:
        raise RuntimeError(f'Prompt excede {MAX_INPUT_TOKENS} tokens: {input_tokens}.')
    inputs = tokenizer(prompt, return_tensors='pt', truncation=False).to(DEVICE)
    started = time.perf_counter()
    import torch
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=96, do_sample=False, num_beams=1)
    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    return {'answer': answer, 'model': FLAN_MODEL_NAME, 'input_tokens': input_tokens,
            'output_tokens': int(output_ids.shape[-1]),
            'latency_seconds': time.perf_counter() - started,
            'generation_config': {'max_new_tokens': 96, 'do_sample': False, 'num_beams': 1}}

def model_without_corpus(question):
    return generate_with_flan_t5(question, '')

print('Gerador local:', FLAN_MODEL_NAME, '| pesos carregados sob demanda; dispositivo:', DEVICE)


## 7 · Validação mecânica de fontes e política de abstenção
Rejeitamos respostas se houver IDs inexistentes, fontes que não foram enviadas, citações não literais, ou fatos sem nenhuma evidência citada. **O sistema não deve tratar um ID citado como prova automática de correção factual.** Esse controle reduz respostas sem procedência, mas não elimina alucinações.

In [ ]:
def clean_public_answer(answer):
    return str(answer or '').strip()

def answer_question(question, top_k=5, generate=False, generator=None, original_question=None):
    if not isinstance(question, str) or not question.strip():
        raise ValueError('Pergunta vazia.')
    user_question = original_question or question
    canonical_question, corrections = canonicalize_question(question)
    retrieved, gate = select_evidence(canonical_question, top_k=top_k)
    language = detect_response_language(user_question) if 'detect_response_language' in globals() else 'pt' if 'detect_response_language' in globals() else 'pt'
    if generate and generator is None:
        tokenizer, _ = load_flan_t5()
        context, used = build_context(retrieved, canonical_question, tokenizer)
    else:
        context, used = build_context(retrieved)
    targets = explicitly_mentioned_ids(canonical_question)
    result = {'question': user_question, 'canonical_question': canonical_question,
              'normalized_query': normalize_query(canonical_question),
              'spelling_corrections': corrections, 'language': language,
              'retriever': 'tfidf_cosine_from_notebook_02',
              'generator': FLAN_MODEL_NAME if generate else None,
              'status': 'retrieval_only' if used else ('scope_limit' if gate else 'insufficient_evidence'),
              'gate_reason': gate, 'answer': None if used else localized_refusal(user_question, reason=gate, target_ids=targets),
              'source_documents': [{key: row.get(key) for key in ('document_id','pokemon_id','pokemon_name','section','score','factual_scope','source_file')} for row in used],
              'supporting_excerpts': [], 'citation_validation': None,
              'context_chars': len(context), 'documents_in_prompt': len(used)}
    if not used or not generate:
        return result if not used else {**result, 'answer': retrieval_answer(used, user_question)}
    if len(targets) > 1 and not set(targets).issubset({int(row['pokemon_id']) for row in used}):
        return {**result, 'status': 'insufficient_evidence', 'answer': REFUSAL}
    provider = generator or generate_with_flan_t5
    try:
        payload = provider(user_question, context)
        answer = payload.get('answer', '') if isinstance(payload, dict) else str(payload)
    except Exception as error:
        return {**result, 'status': 'provider_unavailable', 'answer': retrieval_answer(used, user_question),
                'generation_error': type(error).__name__}
    if not answer.strip() or answer.strip().casefold() in {'não sei', 'nao sei'}:
        return {**result, 'status': 'insufficient_evidence', 'answer': REFUSAL}
    metadata = payload if isinstance(payload, dict) else {}
    return {**result, 'status': 'answered', 'answer': clean_public_answer(answer),
            'supporting_excerpts': [{'document_id': row['document_id'], 'quote': row['content']} for row in used],
            'generation_metadata': {key: value for key, value in metadata.items() if key != 'answer'}}


## 8 · Testes do contrato e fontes selecionadas


## 8.1 · Avaliação do recuperador efetivamente usado

Esta comparação mantém os mesmos qrels do Notebook 02, mas mede separadamente o ranking TF-IDF puro e `select_evidence()`. Resultados com rótulos provisórios são explicitamente exploratórios.


In [ ]:
CHATBOT_RETRIEVAL_METRICS = EVAL_DIR / 'chatbot_retrieval_metrics_by_query.csv'
CHATBOT_RETRIEVAL_SUMMARY = EVAL_DIR / 'chatbot_retrieval_metrics_summary.csv'
CHATBOT_RETRIEVAL_RANKINGS = EVAL_DIR / 'chatbot_retrieval_rankings.csv'
EVAL_KS = (1, 3, 5, 10)
queries_eval = pd.read_csv(EVAL_DIR / 'evaluation_queries.csv', dtype={'query_id': str})
qrels_eval = pd.read_csv(EVAL_DIR / 'evaluation_qrels.csv', dtype={'query_id': str, 'document_id': str})
qrels_by_query_eval = {qid: set(group.loc[group.relevance > 0, 'document_id']) for qid, group in qrels_eval.groupby('query_id')}

def retrieval_metrics(ranked_ids, relevant, k):
    hits = [int(document_id in relevant) for document_id in ranked_ids[:k]]
    return {'hit_at_k': float(any(hits)), 'precision_at_k': sum(hits) / k,
            'recall_at_k': sum(hits) / len(relevant) if relevant else 0.0,
            'mrr_at_k': next((1 / (index + 1) for index, hit in enumerate(hits) if hit), 0.0)}

rows, rankings = [], []
for query in queries_eval.itertuples(index=False):
    relevant = qrels_by_query_eval.get(query.query_id, set())
    if not relevant:
        continue
    raw = retrieve(query.question, top_k=max(EVAL_KS))
    filtered, gate_reason = select_evidence(query.question, top_k=max(EVAL_KS))
    label_set = 'reviewed' if getattr(query, 'review_status', '') == 'reviewed' else 'provisional'
    for method, result in (('tfidf_raw', raw), ('tfidf_filtered', filtered)):
        ranking = [item['document_id'] for item in result]
        rankings.extend({'query_id': query.query_id, 'method': method, 'rank': rank,
                         'document_id': item['document_id'], 'section': item.get('section'),
                         'pokemon_id': item.get('pokemon_id'), 'gate_reason': gate_reason,
                         'label_set': label_set} for rank, item in enumerate(result, 1))
        for k in EVAL_KS:
            rows.append({'query_id': query.query_id, 'method': method, 'k': k,
                         'label_set': label_set, 'gate_reason': gate_reason,
                         'species_coverage': len({item['pokemon_id'] for item in result[:k]}),
                         **retrieval_metrics(ranking, relevant, k)})
chatbot_metrics = pd.DataFrame(rows)
chatbot_rankings = pd.DataFrame(rankings)
chatbot_summary = chatbot_metrics.groupby(['label_set', 'method', 'k'], as_index=False)[
    ['hit_at_k', 'precision_at_k', 'recall_at_k', 'mrr_at_k', 'species_coverage']].mean()
chatbot_metrics.to_csv(CHATBOT_RETRIEVAL_METRICS, index=False, encoding='utf-8')
chatbot_rankings.to_csv(CHATBOT_RETRIEVAL_RANKINGS, index=False, encoding='utf-8')
chatbot_summary.to_csv(CHATBOT_RETRIEVAL_SUMMARY, index=False, encoding='utf-8')
print('Comparação exploratória tfidf_raw × tfidf_filtered:', CHATBOT_RETRIEVAL_SUMMARY)


## 8.1 · Avaliação do recuperador efetivamente usado

Esta comparação mantém os mesmos qrels do Notebook 02, mas mede separadamente o ranking TF-IDF puro e `select_evidence()`. Resultados com rótulos provisórios são explicitamente exploratórios.


In [ ]:
CHATBOT_RETRIEVAL_METRICS = EVAL_DIR / 'chatbot_retrieval_metrics_by_query.csv'
CHATBOT_RETRIEVAL_SUMMARY = EVAL_DIR / 'chatbot_retrieval_metrics_summary.csv'
CHATBOT_RETRIEVAL_RANKINGS = EVAL_DIR / 'chatbot_retrieval_rankings.csv'
EVAL_KS = (1, 3, 5, 10)
queries_eval = pd.read_csv(EVAL_DIR / 'evaluation_queries.csv', dtype={'query_id': str})
qrels_eval = pd.read_csv(EVAL_DIR / 'evaluation_qrels.csv', dtype={'query_id': str, 'document_id': str})
qrels_by_query_eval = {qid: set(group.loc[group.relevance > 0, 'document_id']) for qid, group in qrels_eval.groupby('query_id')}

def retrieval_metrics(ranked_ids, relevant, k):
    hits = [int(document_id in relevant) for document_id in ranked_ids[:k]]
    return {'hit_at_k': float(any(hits)), 'precision_at_k': sum(hits) / k,
            'recall_at_k': sum(hits) / len(relevant) if relevant else 0.0,
            'mrr_at_k': next((1 / (index + 1) for index, hit in enumerate(hits) if hit), 0.0)}

rows, rankings = [], []
for query in queries_eval.itertuples(index=False):
    relevant = qrels_by_query_eval.get(query.query_id, set())
    if not relevant:
        continue
    raw = retrieve(query.question, top_k=max(EVAL_KS))
    filtered, gate_reason = select_evidence(query.question, top_k=max(EVAL_KS))
    label_set = 'reviewed' if getattr(query, 'review_status', '') == 'reviewed' else 'provisional'
    for method, result in (('tfidf_raw', raw), ('tfidf_filtered', filtered)):
        ranking = [item['document_id'] for item in result]
        rankings.extend({'query_id': query.query_id, 'method': method, 'rank': rank,
                         'document_id': item['document_id'], 'section': item.get('section'),
                         'pokemon_id': item.get('pokemon_id'), 'gate_reason': gate_reason,
                         'label_set': label_set} for rank, item in enumerate(result, 1))
        for k in EVAL_KS:
            rows.append({'query_id': query.query_id, 'method': method, 'k': k,
                         'label_set': label_set, 'gate_reason': gate_reason,
                         'species_coverage': len({item['pokemon_id'] for item in result[:k]}),
                         **retrieval_metrics(ranking, relevant, k)})
chatbot_metrics = pd.DataFrame(rows)
chatbot_rankings = pd.DataFrame(rankings)
chatbot_summary = chatbot_metrics.groupby(['label_set', 'method', 'k'], as_index=False)[
    ['hit_at_k', 'precision_at_k', 'recall_at_k', 'mrr_at_k', 'species_coverage']].mean()
chatbot_metrics.to_csv(CHATBOT_RETRIEVAL_METRICS, index=False, encoding='utf-8')
chatbot_rankings.to_csv(CHATBOT_RETRIEVAL_RANKINGS, index=False, encoding='utf-8')
chatbot_summary.to_csv(CHATBOT_RETRIEVAL_SUMMARY, index=False, encoding='utf-8')
print('Comparação exploratória tfidf_raw × tfidf_filtered:', CHATBOT_RETRIEVAL_SUMMARY)


## 8.1 · Avaliação do recuperador efetivamente usado

Esta comparação mantém os mesmos qrels do Notebook 02, mas mede separadamente o ranking TF-IDF puro e `select_evidence()`. Resultados com rótulos provisórios são explicitamente exploratórios.


In [ ]:
CHATBOT_RETRIEVAL_METRICS = EVAL_DIR / 'chatbot_retrieval_metrics_by_query.csv'
CHATBOT_RETRIEVAL_SUMMARY = EVAL_DIR / 'chatbot_retrieval_metrics_summary.csv'
CHATBOT_RETRIEVAL_RANKINGS = EVAL_DIR / 'chatbot_retrieval_rankings.csv'
EVAL_KS = (1, 3, 5, 10)
queries_eval = pd.read_csv(EVAL_DIR / 'evaluation_queries.csv', dtype={'query_id': str})
qrels_eval = pd.read_csv(EVAL_DIR / 'evaluation_qrels.csv', dtype={'query_id': str, 'document_id': str})
qrels_by_query_eval = {qid: set(group.loc[group.relevance > 0, 'document_id']) for qid, group in qrels_eval.groupby('query_id')}

def retrieval_metrics(ranked_ids, relevant, k):
    hits = [int(document_id in relevant) for document_id in ranked_ids[:k]]
    return {'hit_at_k': float(any(hits)), 'precision_at_k': sum(hits) / k,
            'recall_at_k': sum(hits) / len(relevant) if relevant else 0.0,
            'mrr_at_k': next((1 / (index + 1) for index, hit in enumerate(hits) if hit), 0.0)}

rows, rankings = [], []
for query in queries_eval.itertuples(index=False):
    relevant = qrels_by_query_eval.get(query.query_id, set())
    if not relevant:
        continue
    raw = retrieve(query.question, top_k=max(EVAL_KS))
    filtered, gate_reason = select_evidence(query.question, top_k=max(EVAL_KS))
    label_set = 'reviewed' if getattr(query, 'review_status', '') == 'reviewed' else 'provisional'
    for method, result in (('tfidf_raw', raw), ('tfidf_filtered', filtered)):
        ranking = [item['document_id'] for item in result]
        rankings.extend({'query_id': query.query_id, 'method': method, 'rank': rank,
                         'document_id': item['document_id'], 'section': item.get('section'),
                         'pokemon_id': item.get('pokemon_id'), 'gate_reason': gate_reason,
                         'label_set': label_set} for rank, item in enumerate(result, 1))
        for k in EVAL_KS:
            rows.append({'query_id': query.query_id, 'method': method, 'k': k,
                         'label_set': label_set, 'gate_reason': gate_reason,
                         'species_coverage': len({item['pokemon_id'] for item in result[:k]}),
                         **retrieval_metrics(ranking, relevant, k)})
chatbot_metrics = pd.DataFrame(rows)
chatbot_rankings = pd.DataFrame(rankings)
chatbot_summary = chatbot_metrics.groupby(['label_set', 'method', 'k'], as_index=False)[
    ['hit_at_k', 'precision_at_k', 'recall_at_k', 'mrr_at_k', 'species_coverage']].mean()
chatbot_metrics.to_csv(CHATBOT_RETRIEVAL_METRICS, index=False, encoding='utf-8')
chatbot_rankings.to_csv(CHATBOT_RETRIEVAL_RANKINGS, index=False, encoding='utf-8')
chatbot_summary.to_csv(CHATBOT_RETRIEVAL_SUMMARY, index=False, encoding='utf-8')
print('Comparação exploratória tfidf_raw × tfidf_filtered:', CHATBOT_RETRIEVAL_SUMMARY)


In [ ]:
# O FLAN-T5 retorna texto livre; as fontes continuam selecionadas pelo Python.
target = "Quais são os tipos do Charizard?"
rows, gate = select_evidence(target, top_k=5)
context, used = build_context(rows)
assert gate == "ready" and used
assert used[0]["document_id"] in context
assert token_count is not None and "Resposta:" in prompt_text(target, context)
print("Seleção de fontes, contexto com orçamento e contrato local: OK.")


## 9 · Demonstração do RAG com FLAN-T5


In [ ]:
DEMO_QUESTION = 'Qual é a velocidade-base do Pikachu?'
RUN_LIVE_DEMO = False
if RUN_LIVE_DEMO:
    print(answer_question(DEMO_QUESTION, generate=True))
else:
    print('Demonstração FLAN-T5 desativada. Altere RUN_LIVE_DEMO=True para baixar/carregar o modelo.')


## 10 · Conversa contextual com memória curta
Uma sessão guarda os Pokémon mencionados (inclusive em perguntas com duas espécies), reformula seguimentos antes do TF-IDF e solicita esclarecimento quando um pronome singular é ambíguo. Não usa respostas passadas como fonte factual.


In [ ]:
class PokedexChatSession:
    """Memória curta de referências; fatos sempre vêm do TF-IDF + documentos."""
    def __init__(self):
        self.active_ids = []
        self.last_section = None
        self.last_question = None
        self.pending_question = None
        self.pending_candidates = []
        self.turns = []

    def _is_followup(self, question):
        if not self.active_ids or explicitly_mentioned_ids(question):
            return False
        q = normalize_plain(question)
        if len(q.split()) > 12:
            return False
        patterns = [
            r"\b(ele|ela|dele|dela|deles|delas|seu|sua|seus|suas|ambos|ambas)\b",
            r"\b(it|its|they|them|their|both)\b",
            r"^(?:e |and )?(?:qual|quais|what|which|how|where|when|como|onde|quando|de quem|para quem)\b",
            r"^(?:e |and )?(?:em que nivel|qual nivel|at what level)\b",
            r"^(?:e |and )?(?:o |a |os |as )?(?:tipo|tipos|fraqueza|evolucao|nivel|type|types|evolution|level)\b"
        ]
        return any(re.search(p, q) for p in patterns)

    @staticmethod
    def _is_plural_reference(question):
        q = normalize_plain(question)
        return bool(re.search(r"\b(eles|elas|deles|delas|ambos|ambas|dos dois|das duas|they|them|their|both|each)\b", q))

    @staticmethod
    def _is_singular_reference(question):
        q = normalize_plain(question)
        return bool(re.search(r"\b(dele|dela|ele|ela|seu|sua|it|its|him|her)\b", q))

    def _clarification(self, question):
        en = detect_response_language(question) == "en"
        names = pokemon_names_by_ids(self.active_ids)
        if len(names) == 2:
            options = f"{names[0]} or {names[1]}" if en else f"{names[0]} ou {names[1]}"
        else:
            options = ", ".join(names)
        message = (f"Which Pokémon do you mean: {options}?" if en
                   else f"De qual Pokémon você está falando: {options}?")
        self.pending_question = question
        self.pending_candidates = list(self.active_ids)
        return {
            "question": question, "original_question": question,
            "resolved_question": None, "context_used": True,
            "active_pokemon": names, "language": "en" if en else "pt",
            "status": "clarification_needed", "gate_reason": "ambiguous_reference",
            "answer": message, "source_documents": [], "citation_validation": None,
            "returned_citations": [], "supporting_excerpts": []
        }

    def _resolve(self, question, target_ids):
        en = detect_response_language(question) == "en"
        names = pokemon_names_by_ids(target_ids)
        joined = " and ".join(names) if en else " e ".join(names)
        q = normalize_plain(question)
        section = requested_section(question)
        if section is None and re.search(r"\b(nivel|level|pedra|stone)\b", q):
            section = "evolution"
        if section is None:
            section = self.last_section or "profile"
        if section == "evolution":
            if self.last_section in {"encounters", "obtainment"} and re.search(r"\b(evolucao|evolution|evolui|evolve|nivel|level)\b", q):
                return (f"Which Pokémon evolves into {joined} and under what conditions?" if en
                        else f"De qual Pokémon {joined} evolui e em quais condições?"), section
            return (f"What is the evolution line of {joined}, including conditions and levels?" if en
                    else f"Qual é a linha evolutiva de {joined}, incluindo condições e níveis?"), section
        if section == "profile" and re.search(r"\b(tipo|tipos|type|types)\b", q):
            return (f"What are the types of {joined}?" if en else f"Quais são os tipos de {joined}?"), section
        # Preserva as palavras do usuário, acrescentando o referente sem tags artificiais.
        return (f"{question.strip()} (about {joined})" if en else
                f"{question.strip()} (sobre {joined})"), section

    def _remember(self, question, target_ids, section, result):
        self.active_ids = list(target_ids)
        self.last_section = section
        self.last_question = question
        self.turns.append(result)
        self.turns = self.turns[-8:]

    def ask(self, question, generate=False, generator=None, top_k=5):
        if not isinstance(question, str) or not question.strip():
            raise ValueError("Pergunta vazia.")
        question = question.strip()
        explicit = explicitly_mentioned_ids(question)
        resolved = question
        context_used = False
        targets = list(explicit)

        # Permite responder "Squirtle" após um pedido de esclarecimento.
        if self.pending_question is not None:
            if len(explicit) == 1 and explicit[0] in self.pending_candidates and len(normalize_plain(question).split()) <= 3:
                pending = self.pending_question
                self.pending_question = None
                self.pending_candidates = []
                resolved, section = self._resolve(pending, explicit)
                context_used = True
                targets = list(explicit)
            else:
                self.pending_question = None
                self.pending_candidates = []
                section = requested_section(question) or "profile"
        else:
            section = requested_section(question) or "profile"

        if not context_used:
            if explicit:
                targets = list(explicit)
            elif self._is_followup(question):
                if len(self.active_ids) > 1 and not self._is_plural_reference(question):
                    # Em contexto com duas espécies, "dele" e "qual evolução?" são ambíguos.
                    result = self._clarification(question)
                    self.turns.append(result)
                    self.turns = self.turns[-8:]
                    return result
                targets = list(self.active_ids)
                resolved, section = self._resolve(question, targets)
                context_used = True
            else:
                # Pergunta nova sem referência explícita: evita vazamento de contexto.
                targets = []

        result = answer_question(
            resolved, top_k=top_k, generate=generate,
            generator=generator, original_question=question
        )
        result["original_question"] = question
        result["resolved_question"] = resolved
        result["context_used"] = context_used
        result["active_pokemon"] = pokemon_names_by_ids(targets)
        self._remember(question, targets, section, result)
        return result


session = PokedexChatSession()
print("Sessão conversacional pronta: múltiplos Pokémon, pronomes e esclarecimentos.")


### Testes de continuidade e ambiguidade (sem FLAN-T5)
Estes testes validam referências e recuperação. Execute-os após atualizar o índice do Notebook 02.


In [ ]:
# Contratos de seleção independentes da ordem de definição das células.
assert requested_section("Como encontrar o Eevee?") == "encounters"
assert requested_section("Como evoluir o Charizard?") == "evolution"
rows, gate = select_evidence("Quais os pokemons com o mesmo tipo do charizard?", top_k=5)
assert gate == "ready" and {6, 146}.issubset({int(row["pokemon_id"]) for row in rows})
print("Contratos de intenção, fontes e comparação por tipo: OK.")


## 11 · Interface de chat opcional no Colab
Execute `open_chat_ui()` para abrir o chat. Marque `Gerar com FLAN-T5 local` para usar o modelo; não é necessária chave de API. A interface exibe também os documentos recuperados. Uma futura UI Next.js poderá consumir essa lógica via FastAPI.

In [ ]:
def open_chat_ui():
    try:
        import ipywidgets as widgets
    except ImportError:
        print('Instale ipywidgets para abrir a interface.')
        return None
    conversation = PokedexChatSession()
    field = widgets.Text(placeholder='Ex.: Como Eevee evolui?', description='Pergunta', layout=widgets.Layout(width='82%'))
    action = widgets.Button(description='Perguntar', button_style='primary')
    do_gen = widgets.Checkbox(value=False, description='Gerar com FLAN-T5 local')
    output = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='8px'))
    hint = widgets.HTML('<b>FLAN-T5 local.</b> A primeira geração pode baixar os pesos do Hugging Face.')
    def run(_):
        prompt = field.value.strip()
        if not prompt: return
        field.value = ''
        with output:
            try: show_answer(conversation.ask(prompt, generate=bool(do_gen.value)))
            except Exception as error: print('Falha na geração local:', type(error).__name__)
    action.on_click(run)
    display(widgets.VBox([hint, widgets.HBox([field, action]), do_gen, output]))
    return conversation


## 12 · Avaliação do gerador (separada da recuperação)
O Notebook 02 já mediu Precision@K, Recall@K e MRR. Aqui registramos **abstenção, cobertura de citações, integridade das evidências** e uma **planilha de revisão humana** sobre correção factual, suporte semântico e completude.

- As perguntas são apenas um conjunto inicial: revise e amplie antes de reportar resultados definitivos.
- **A avaliação com FLAN-T5 é opt-in** por causa de consumo de API.
- Não calculamos automaticamente *fidelidade factual*: verificar substring não é provar que a afirmação está correta.
- Os JSONLs exportados registram respostas e IDs, mas nunca a chave de API.

In [ ]:
EVAL_QUESTIONS_PATH = EVAL_DIR / 'rag_generation_questions.csv'
EVAL_RESULTS_PATH = EVAL_DIR / 'rag_generation_results.jsonl'
EVAL_REVIEW_PATH = EVAL_DIR / 'rag_manual_review.csv'
seed_cases = [
    ('R01', 'Quais são os tipos do Charizard?', 'answerable', 'profile'),
    ('R02', 'Qual é a velocidade-base do Pikachu?', 'answerable', 'stats'),
    ('R03', 'Compare a velocidade-base de Charizard e Blastoise.', 'comparison', 'stats'),
    ('R04', 'Quais ginásios preciso vencer?', 'out_of_scope', 'none'),
    ('R05', 'Qual é o número do sapato do Pikachu?', 'missing_evidence', 'profile'),
]
if not EVAL_QUESTIONS_PATH.exists():
    pd.DataFrame(seed_cases, columns=['query_id','question','expected_case','expected_section']).to_csv(
        EVAL_QUESTIONS_PATH, index=False, encoding='utf-8')
questions = pd.read_csv(EVAL_QUESTIONS_PATH)
assert questions['query_id'].is_unique

def precheck_generation_case(row):
    selected, gate = select_evidence(row.question, top_k=5)
    corpus_text = '\n'.join(str(document['content']).casefold() for document in documents)
    if row.expected_case in {'answerable', 'comparison'}:
        assert selected, f'{row.query_id}: sem evidência recuperada para caso respondível'
    elif row.expected_case == 'out_of_scope':
        assert gate == 'out_of_scope', f'{row.query_id}: deveria ser fora de escopo'
    elif row.expected_case == 'missing_evidence':
        assert 'sapato' not in corpus_text and 'shoe' not in corpus_text, 'Caso ausente exige inspeção do corpus.'
    return {'query_id': row.query_id, 'precheck_gate_reason': gate,
            'precheck_source_ids': [item['document_id'] for item in selected]}

prechecks = [precheck_generation_case(row) for row in questions.itertuples(index=False)]
pd.DataFrame(prechecks).to_csv(EVAL_DIR / 'rag_generation_precheck.csv', index=False, encoding='utf-8')
RUN_GENERATION_EVALUATION = False
if RUN_GENERATION_EVALUATION:
    results = []
    for row in questions.itertuples(index=False):
        record = {'query_id': row.query_id, 'question': row.question, 'expected_case': row.expected_case}
        try:
            record['without_rag_answer'] = model_without_corpus(row.question)
        except Exception as error:
            record['without_rag_error'] = type(error).__name__
        try:
            grounded = answer_question(row.question, generate=True)
            record.update(with_rag=grounded, status=grounded['status'],
                          source_documents=grounded['source_documents'],
                          citations=grounded['returned_citations'])
        except Exception as error:
            record.update(status='api_error', api_error=type(error).__name__)
        results.append(record)
    with EVAL_RESULTS_PATH.open('w', encoding='utf-8') as handle:
        for result in results:
            handle.write(json.dumps(result, ensure_ascii=False) + '\n')
    print('Avaliação real opt-in exportada:', EVAL_RESULTS_PATH)
else:
    print('Avaliação real desativada: RUN_GENERATION_EVALUATION=False; nenhuma chamada ao FLAN-T5 foi feita.')

if EVAL_RESULTS_PATH.exists():
    review_rows = []
    for result in read_jsonl(EVAL_RESULTS_PATH):
        review_rows.append({'query_id': result.get('query_id'), 'question': result.get('question'),
                            'status': result.get('status'), 'fidelity_to_text_0_2': '',
                            'relevance_0_2': '', 'completeness_0_2': '',
                            'factual_accuracy_0_2': '', 'appropriate_abstention_0_2': '',
                            'reviewer_notes': '', 'reviewed_by': '', 'review_date': ''})
    pd.DataFrame(review_rows).to_csv(EVAL_REVIEW_PATH, index=False, encoding='utf-8')
    print('Planilha de revisão humana:', EVAL_REVIEW_PATH)


## 13 · Experimentos ablação e contrafactual (opcionais, isolados da produção)
**Experimento A — Sem contexto**: mede qualitativamente como um LLM responde sem nosso corpus. **Não faz parte do chatbot**.

**Experimento B — Contrafactual sintético**: envia um documento fictício rotulado como teste somente na memória, sem modificar os JSONs reais, para investigar dependência do contexto. Também não altera o recuperador.

Esses experimentos são desativados por padrão e exigem revisão humana para conclusões.

In [ ]:
ABLATION_DIR = EVAL_DIR / 'ablations'
ABLATION_DIR.mkdir(parents=True, exist_ok=True)
RUN_ABLATION_WITHOUT_CONTEXT = False
RUN_SYNTHETIC_COUNTERFACTUAL = False

if RUN_ABLATION_WITHOUT_CONTEXT:
    sample = 'Qual é a velocidade-base do Pikachu?'
    comparison = {'experiment': 'without_context_vs_rag', 'question': sample,
                  'baseline_without_corpus': model_without_corpus(sample),
                  'rag': answer_question(sample, generate=True),
                  'interpretation': 'Requer revisão humana; não supõe superioridade do RAG.'}
    (ABLATION_DIR / 'without_context_vs_rag.json').write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding='utf-8')
if RUN_SYNTHETIC_COUNTERFACTUAL:
    raise RuntimeError('Experimento sintético deve ser implementado somente com um protocolo revisado manualmente.')
print('Experimentos remotos opcionais permanecem desativados por padrão.')


## 14 · Relatório de integridade e limitações
O relatório diferencia testes determinísticos locais de geração real. Não declare que a fidelidade do modelo foi comprovada sem executar a API e revisar manualmente a planilha.

In [ ]:
report = {
 'project':'Pokédex Inteligente — Trabalho 2', 'notebook':'03_RAG_CHATBOT', 'run_utc':RUN_AT,
 'document_count':len(documents), 'pokemon_count':len(records),
 'retriever':'TF-IDF + cosseno persistido no Notebook 02; sem novo fit',
 'generator':'FLAN-T5 local', 'model':FLAN_MODEL_NAME, 'model_loaded':FLAN_MODEL is not None,
 'input_token_budget':MAX_INPUT_TOKENS, 'live_demo_executed':RUN_LIVE_DEMO,
 'generation_evaluation_executed':RUN_GENERATION_EVALUATION,
 'limitations':['FLAN-T5-small pode ser menos consistente em português.',
                'A seleção de fontes e a citação textual não comprovam suporte semântico sozinhas.',
                'A primeira inferência exige acesso ao Hugging Face para baixar os pesos.'],
}
report_path=REPORTS_DIR/'rag_chatbot_grounding_report.json'
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
assert actual_sha==manifest['source_document_sha256']
assert index_matrix.shape[0]==len(documents)==len(manifest['document_ids'])
print('Relatório:',report_path)


## 15 · Síntese para a apresentação
1. **PLN tradicional**: corpus coletado e preparado no Trabalho 1, representado em documentos no Notebook 01 e indexado com TF-IDF no Notebook 02.
2. **Recuperação verificável**: toda pergunta passa pelo recuperador tradicional e pelos controles de escopo/cobertura antes da geração.
3. **LLM como etapa final**: FLAN-T5 apenas formula a resposta a partir do contexto recuperado, sem ferramentas de busca externa.
4. **Fundamentação**: cada resposta aceita traz IDs válidos e trechos literais localizados nos documentos enviados; a interpretação semântica é revisada separadamente.
5. **Avaliação**: Notebook 02 mede recuperação, Notebook 03 mede integridade das fontes, qualidade das respostas e abstenções, com experimentos opcionais de ablação.

**Limitação central:** RAG com verificação de citações não é garantia absoluta de verdade. A validação final depende do conteúdo e da qualidade histórica do corpus.